# CaST-POI: Candidate-Aware Spatiotemporal Modeling for Next POI Recommendation

This notebook implements the complete CaST-POI model as described in the manuscript, including:
- **Real dataset download and preprocessing** (NYC, TKY, CA)
- **Full model implementation** with candidate-conditioned attention
- **Ablation study** (Table 3)
- **Candidate pool size experiment** (Figure 3)
- **Robustness analysis** (Figure 4)
- **Efficiency evaluation** (Table 4)

**Core Innovation**: Candidate-conditioned sequence reader that dynamically attends to different historical behaviors for different candidate POIs.

**Key Formula**:
$$\alpha_i^{(c)} \propto \exp\left(\frac{\mathbf{q}_c^\top \mathbf{k}_i}{\sqrt{d}} + b_t(\Delta t_i) + b_s(\Delta d_i(c))\right)$$

---
## Part 1: Setup & Configuration

In [1]:
# Section 1: Setup & Dependencies
# Uncomment if running on Colab
# !pip install torch numpy pandas matplotlib tqdm requests -q

import math
import time
import random
import requests
import io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Optional, Dict, List, Tuple
from collections import defaultdict
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# 🔧 DATASET CONFIGURATION - MODIFY THIS TO SWITCH DATASETS
# ============================================================
# Options: 'nyc', 'tky', 'ca'
DATASET = 'nyc'  # <-- Change this to run on different datasets
# ============================================================

USE_REAL_DATA = True  # True: download real data, False: use synthetic data
QUICK_MODE = False  # True: reduced epochs for quick testing

# Section 2: Configuration
CONFIG = {
    # Model (from manuscript Section 5.1.4)
    'embed_dim': 64,
    'num_heads': 8,
    'num_layers': 2,
    'ff_dim': 256,
    'max_seq_len': 50,
    'dropout': 0.1,

    # Data (from manuscript Section 5.1.1)
    'max_history_len': 50,
    'num_negatives': 99,  # 1 positive + 99 negatives = 100 candidates (training only)
    'min_user_checkins': 10,
    'min_poi_checkins': 10,
    'test_size': 30,  # Last 30 check-ins for test

    # Training (from manuscript Section 5.1.4)
    'batch_size': 64,
    'num_epochs': 50,
    'learning_rate': 0.001,  # Paper: "learning rate 10^{-3}"
    'weight_decay': 0.0001,
    'gradient_clip': 5.0,
    'early_stopping_patience': 10,
    'label_smoothing': 0.1,
    'explore_weight': 3.0,
    'warmup_epochs': 3,

    # Evaluation (from manuscript)
    'eval_batch_size': 16,
    'eval_ks': [5, 10, 20],
    'candidate_pool_sizes': [50, 100, 200, 500],
}

if QUICK_MODE:
    CONFIG['num_epochs'] = 10
    CONFIG['early_stopping_patience'] = 3

# ============================================================
# Dataset-specific hyperparameter overrides
# ============================================================
DATASET_OVERRIDES = {
    'nyc': {},  # defaults work well for NYC
    'tky': {
        'learning_rate': 0.0005,
        'dropout': 0.15,
        'warmup_epochs': 5,
        'num_negatives': 199,
        'early_stopping_patience': 12,
        'explore_weight': 4.0,
        'dist_buckets': [0, 0.1, 0.5, 2, 10, float('inf')],  # default
    },
    'ca': {
        'learning_rate': 0.0005,
        'dropout': 0.2,
        'warmup_epochs': 5,
        'num_negatives': 199,
        'early_stopping_patience': 15,
        'explore_weight': 5.0,
        'dist_buckets': [0, 1, 5, 20, 100, float('inf')],  # wider for CA's large geo spread
    },
}

_overrides = DATASET_OVERRIDES.get(DATASET, {})
CONFIG.update(_overrides)

print(f"\n{'='*60}")
print(f"RUNNING EXPERIMENTS ON: {DATASET.upper()}")
print('='*60)
print("\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Configuration:
  embed_dim: 64
  num_heads: 8
  num_layers: 2
  ff_dim: 256
  max_seq_len: 50
  dropout: 0.1
  max_history_len: 50
  num_negatives: 99
  min_user_checkins: 10
  min_poi_checkins: 10
  test_size: 30
  batch_size: 64
  num_epochs: 50
  learning_rate: 0.001
  weight_decay: 0.0001
  gradient_clip: 5.0
  early_stopping_patience: 10
  label_smoothing: 0.1
  explore_weight: 3.0
  warmup_epochs: 3
  eval_batch_size: 16
  eval_ks: [5, 10, 20]
  candidate_pool_sizes: [50, 100, 200, 500]


---
## Part 2: Data Download & Preprocessing

In [3]:
# Section 3: Dataset Download (from download.py)

# Dataset URLs - Using HuggingFace LLM4POI preprocessed data (same source as CoMaPOI)
DATASET_INFO = {
    'nyc': {
        'url': 'https://huggingface.co/datasets/w11wo/LLM4POI/resolve/main/nyc/preprocessed/train_sample.csv',
        'stats': {'users': 988, 'pois': 5086, 'checkins': 99964},
        'lat_range': (40.5, 41.0),
        'lon_range': (-74.3, -73.7)
    },
    'tky': {
        'url': 'https://huggingface.co/datasets/w11wo/LLM4POI/resolve/main/tky/preprocessed/train_sample.csv',
        'stats': {'users': 2206, 'pois': 7849, 'checkins': 325313},
        'lat_range': (35.5, 36.0),
        'lon_range': (139.5, 140.0)
    },
    'ca': {
        'url': 'https://huggingface.co/datasets/w11wo/LLM4POI/resolve/main/ca/preprocessed/train_sample.csv',
        'stats': {'users': 1818, 'pois': 13564, 'checkins': 174791},
        'lat_range': (33.5, 38.5),
        'lon_range': (-122.5, -117.0)
    }
}


def download_dataset(dataset_name: str) -> pd.DataFrame:
    """Download real Foursquare dataset from HuggingFace LLM4POI repository."""
    dataset_name = dataset_name.lower()
    if dataset_name not in DATASET_INFO:
        raise ValueError(f"Unknown dataset: {dataset_name}. Choose from {list(DATASET_INFO.keys())}")

    url = DATASET_INFO[dataset_name]['url']
    print(f"Downloading {dataset_name.upper()} dataset from HuggingFace...")

    try:
        response = requests.get(url, timeout=120, allow_redirects=True)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))
        print(f"Downloaded {len(df)} records, columns: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"Download failed: {e}")
        print("Falling back to synthetic data...")
        return create_synthetic_data(dataset_name)


def create_synthetic_data(dataset_name: str) -> pd.DataFrame:
    """Create synthetic data matching real dataset statistics."""
    dataset_name = dataset_name.lower()
    info = DATASET_INFO.get(dataset_name, DATASET_INFO['nyc'])
    stats = info['stats']

    print(f"Creating synthetic {dataset_name.upper()} dataset...")

    n_checkins = min(stats['checkins'], 100000)  # Limit for memory
    n_users = stats['users']
    n_pois = stats['pois']

    np.random.seed(42)

    # Generate user IDs with realistic distribution (some users more active)
    user_weights = np.random.pareto(1.5, n_users) + 1
    user_weights = user_weights / user_weights.sum()
    user_ids = np.random.choice(n_users, n_checkins, p=user_weights)

    # Generate POI IDs with realistic distribution (some POIs more popular)
    poi_weights = np.random.pareto(1.2, n_pois) + 1
    poi_weights = poi_weights / poi_weights.sum()
    poi_ids = np.random.choice(n_pois, n_checkins, p=poi_weights)

    # Generate coordinates
    lat_range, lon_range = info['lat_range'], info['lon_range']
    latitudes = np.random.uniform(lat_range[0], lat_range[1], n_checkins)
    longitudes = np.random.uniform(lon_range[0], lon_range[1], n_checkins)

    # Generate timestamps
    base_time = 1334000000  # April 2012
    timestamps = base_time + np.sort(np.random.randint(0, 10 * 30 * 24 * 3600, n_checkins))

    # Generate categories
    categories = np.random.choice(
        ['Food', 'Shop', 'Travel', 'Arts', 'Outdoors', 'Nightlife', 'College', 'Professional'],
        n_checkins
    )

    df = pd.DataFrame({
        'user_id': user_ids,
        'poi_id': poi_ids,
        'latitude': latitudes,
        'longitude': longitudes,
        'timestamp': timestamps,
        'category': categories
    })

    # Sort by user and timestamp
    df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
    print(f"Created {len(df)} synthetic records")

    return df

In [4]:
# Section 4: Data Preprocessing (from preprocess.py)

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names to: user_id, poi_id, latitude, longitude, timestamp, category."""
    column_mapping = {
        # HuggingFace LLM4POI format
        'UserId': 'user_id',
        'PoiId': 'poi_id',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'UTCTimeOffsetEpoch': 'timestamp',
        'UTCTimeOffset': 'datetime_str',
        'PoiCategoryName': 'category',
        'PoiCategoryId': 'category_id',
        # Other common formats
        'userid': 'user_id', 'user': 'user_id',
        'venueId': 'poi_id', 'venue_id': 'poi_id', 'poi': 'poi_id',
        'venueCategoryId': 'category_id',
        'venueCategory': 'category',
        'lat': 'latitude', 'lng': 'longitude', 'lon': 'longitude',
        'utcTimestamp': 'timestamp', 'time': 'timestamp'
    }
    df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})

    # Ensure timestamp is numeric (Unix epoch)
    if 'timestamp' in df.columns:
        if df['timestamp'].dtype == object:
            # Convert datetime string to Unix timestamp
            df['timestamp'] = pd.to_datetime(df['timestamp']).astype(int) // 10**9
        else:
            df['timestamp'] = df['timestamp'].astype(float).astype(int)

    # Keep only needed columns
    needed = ['user_id', 'poi_id', 'latitude', 'longitude', 'timestamp', 'category']
    available = [c for c in needed if c in df.columns]
    df = df[available].copy()

    print(f"Standardized columns: {list(df.columns)}")
    print(f"  Timestamp range: {df['timestamp'].min()} - {df['timestamp'].max()}")
    print(f"  Lat range: {df['latitude'].min():.4f} - {df['latitude'].max():.4f}")
    print(f"  Lon range: {df['longitude'].min():.4f} - {df['longitude'].max():.4f}")

    return df


def filter_data(df: pd.DataFrame, min_user_checkins: int = 10, min_poi_checkins: int = 10) -> pd.DataFrame:
    """Iteratively filter users and POIs with few check-ins."""
    print(f"Original data: {len(df)} check-ins, {df['user_id'].nunique()} users, {df['poi_id'].nunique()} POIs")

    prev_len = 0
    iteration = 0

    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1

        # Filter users
        user_counts = df['user_id'].value_counts()
        valid_users = user_counts[user_counts >= min_user_checkins].index
        df = df[df['user_id'].isin(valid_users)]

        # Filter POIs
        poi_counts = df['poi_id'].value_counts()
        valid_pois = poi_counts[poi_counts >= min_poi_checkins].index
        df = df[df['poi_id'].isin(valid_pois)]

    print(f"After filtering: {len(df)} check-ins, {df['user_id'].nunique()} users, {df['poi_id'].nunique()} POIs")
    return df.reset_index(drop=True)


def create_id_mappings(df: pd.DataFrame) -> Tuple[Dict, Dict]:
    """Create user_id and poi_id to index mappings.
    POI indices start from 1 (index 0 is reserved for padding)."""
    users = sorted(df['user_id'].unique())
    pois = sorted(df['poi_id'].unique())
    user2idx = {u: i for i, u in enumerate(users)}
    # Start POI indices from 1 to reserve 0 for padding
    poi2idx = {p: i + 1 for i, p in enumerate(pois)}
    return user2idx, poi2idx


def create_poi_info(df: pd.DataFrame, poi2idx: Dict) -> pd.DataFrame:
    """Create POI information table."""
    poi_info = df.groupby('poi_id').agg({
        'latitude': 'mean',
        'longitude': 'mean',
        'category': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
    }).reset_index()
    poi_info['poi_idx'] = poi_info['poi_id'].map(poi2idx)
    return poi_info.sort_values('poi_idx').reset_index(drop=True)


def create_user_trajectories(df: pd.DataFrame, user2idx: Dict, poi2idx: Dict) -> Dict[int, List[Dict]]:
    """Create user trajectories sorted by timestamp."""
    trajectories = defaultdict(list)
    df = df.sort_values(['user_id', 'timestamp'])

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Creating trajectories"):
        user_idx = user2idx[row['user_id']]
        poi_idx = poi2idx[row['poi_id']]
        trajectories[user_idx].append({
            'poi_idx': poi_idx,
            'timestamp': row['timestamp'],
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'category': row.get('category', 'Unknown')
        })

    return dict(trajectories)


def split_train_test(trajectories: Dict[int, List[Dict]], test_size: int = 30) -> Tuple[Dict, Dict]:
    """Split trajectories into train and test. Test set: last test_size check-ins per user."""
    train_data, test_data = {}, {}

    for user_idx, traj in trajectories.items():
        if len(traj) <= test_size:
            continue  # Skip users with too few check-ins
        train_data[user_idx] = traj[:-test_size]
        test_data[user_idx] = traj[-test_size:]

    print(f"Train users: {len(train_data)}, Test users: {len(test_data)}")
    return train_data, test_data

In [5]:
# Section 5: Data Loading Function

def load_dataset(dataset_name: str = 'nyc', use_real_data: bool = True, config: Dict = None) -> Dict:
    """
    Main entry function for loading and preprocessing dataset.

    Args:
        dataset_name: 'nyc', 'tky', or 'ca'
        use_real_data: True to download real data, False for synthetic
        config: Configuration dictionary

    Returns:
        Dictionary containing all processed data
    """
    if config is None:
        config = CONFIG

    print(f"\n{'='*60}")
    print(f"Loading {dataset_name.upper()} dataset (real_data={use_real_data})")
    print('='*60)

    # Load data
    if use_real_data:
        df = download_dataset(dataset_name)
    else:
        df = create_synthetic_data(dataset_name)

    # Standardize columns
    df = standardize_columns(df)

    # Filter data
    df = filter_data(df, config['min_user_checkins'], config['min_poi_checkins'])

    # Create mappings (POI indices start from 1, 0 is reserved for padding)
    user2idx, poi2idx = create_id_mappings(df)
    num_pois = len(poi2idx) + 1  # +1 for padding index 0
    print(f"Final: {len(user2idx)} users, {len(poi2idx)} POIs (embedding size: {num_pois})")

    # Create POI info
    poi_info = create_poi_info(df, poi2idx)

    # Create trajectories
    trajectories = create_user_trajectories(df, user2idx, poi2idx)

    # Split train/test
    train_data, test_data = split_train_test(trajectories, config['test_size'])

    # Calculate statistics
    train_lengths = [len(t) for t in train_data.values()]
    test_lengths = [len(t) for t in test_data.values()]

    stats = {
        'dataset': dataset_name.upper(),
        'num_users': len(user2idx),
        'num_pois': num_pois,  # Includes padding index 0
        'num_real_pois': len(poi2idx),
        'num_checkins': len(df),
        'train_users': len(train_data),
        'test_users': len(test_data),
        'avg_train_length': np.mean(train_lengths) if train_lengths else 0,
        'avg_test_length': np.mean(test_lengths) if test_lengths else 0
    }

    print(f"\nDataset Statistics:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

    return {
        'user2idx': user2idx,
        'poi2idx': poi2idx,
        'poi_info': poi_info,
        'train_data': train_data,
        'test_data': test_data,
        'stats': stats
    }


# Load dataset
data = load_dataset(DATASET, USE_REAL_DATA, CONFIG)


Loading NYC dataset (real_data=True)
Downloaded 83228 records, columns: ['check_ins_id', 'UTCTimeOffset', 'UTCTimeOffsetEpoch', 'pseudo_session_trajectory_id', 'UserId', 'Latitude', 'Longitude', 'PoiId', 'PoiCategoryId', 'PoiCategoryName', 'trajectory_id']
Standardized columns: ['user_id', 'poi_id', 'latitude', 'longitude', 'timestamp', 'category']
  Timestamp range: 1333425609 - 1352439511
  Lat range: 40.5573 - 40.9876
  Lon range: -74.2708 - -73.6862
Original data: 83228 check-ins, 1047 users, 4980 POIs
After filtering: 68305 check-ins, 770 users, 2590 POIs
Final: 770 users, 2590 POIs (embedding size: 2591)


Creating trajectories:   0%|          | 0/68305 [00:00<?, ?it/s]

Train users: 498, Test users: 498

Dataset Statistics:
  dataset: NYC
  num_users: 770
  num_pois: 2591
  num_real_pois: 2590
  num_checkins: 68305
  train_users: 498
  test_users: 498
  avg_train_length: 96.92771084337349
  avg_test_length: 30.0


---
## Part 3: Model Components

In [6]:
# Section 6: Embedding Layers (from embeddings.py)

class POIEmbedding(nn.Module):
    """POI embedding layer with optional dropout."""
    def __init__(self, num_pois: int, embed_dim: int, padding_idx: int = 0, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(num_pois, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        nn.init.normal_(self.embedding.weight, mean=0, std=0.02)
        if padding_idx is not None:
            self.embedding.weight.data[padding_idx].zero_()

    def forward(self, poi_ids: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.embedding(poi_ids))


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    def __init__(self, embed_dim: int, max_len: int = 500, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, :x.size(1)])


class TemporalEncoding(nn.Module):
    """
    Temporal encoding based on hour-of-day (24 bins) and day-of-week (7 bins).
    From paper Section 4.2: "discretizes hour-of-day into 24 bins and day-of-week
    into 7 bins, with separate learnable embeddings for each bin that are summed."
    """
    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        self.hour_embedding = nn.Embedding(24, embed_dim)
        self.dow_embedding = nn.Embedding(7, embed_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.normal_(self.hour_embedding.weight, mean=0, std=0.02)
        nn.init.normal_(self.dow_embedding.weight, mean=0, std=0.02)

    def forward(self, timestamps: torch.Tensor, reference_time: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            timestamps: [B, L] Unix timestamps (float)
        Returns:
            [B, L, D] temporal embeddings
        """
        # Convert Unix timestamps to hour-of-day and day-of-week
        # Use integer division to extract time features
        ts_long = timestamps.long()
        # hour of day: (ts % 86400) / 3600
        hour_of_day = ((ts_long % 86400) // 3600).clamp(0, 23)
        # day of week: ((ts / 86400) + 4) % 7  (Unix epoch was Thursday=4)
        day_of_week = ((ts_long // 86400 + 4) % 7).clamp(0, 6)

        hour_emb = self.hour_embedding(hour_of_day)
        dow_emb = self.dow_embedding(day_of_week)
        return self.dropout(hour_emb + dow_emb)


class SpatialEncoding(nn.Module):
    """Spatial encoding based on GPS coordinates using MLP.
    From paper: "processes raw GPS coordinates through a two-layer MLP with ReLU activation"."""
    def __init__(self, embed_dim: int, dropout: float = 0.1,
                 lat_mean: float = 40.0, lon_mean: float = -74.0,
                 lat_std: float = 1.0, lon_std: float = 1.0):
        super().__init__()
        self.lat_mean = lat_mean
        self.lon_mean = lon_mean
        self.lat_std = lat_std
        self.lon_std = lon_std
        self.projection = nn.Sequential(
            nn.Linear(2, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, embed_dim), nn.Dropout(dropout)
        )

    def forward(self, locations: torch.Tensor) -> torch.Tensor:
        normalized = locations.clone()
        normalized[..., 0] = (locations[..., 0] - self.lat_mean) / self.lat_std
        normalized[..., 1] = (locations[..., 1] - self.lon_mean) / self.lon_std
        return self.projection(normalized)


class CombinedEmbedding(nn.Module):
    """Combined embedding: POI + Positional + Temporal + Spatial."""
    def __init__(self, num_pois: int, embed_dim: int, max_len: int = 100,
                 use_temporal: bool = True, use_spatial: bool = True, dropout: float = 0.1,
                 lat_mean: float = 40.0, lon_mean: float = -74.0,
                 lat_std: float = 1.0, lon_std: float = 1.0):
        super().__init__()
        self.poi_embed = POIEmbedding(num_pois, embed_dim, dropout=dropout)
        self.pos_embed = PositionalEncoding(embed_dim, max_len, dropout=dropout)
        self.use_temporal, self.use_spatial = use_temporal, use_spatial
        if use_temporal:
            self.temporal_embed = TemporalEncoding(embed_dim, dropout=dropout)
        if use_spatial:
            self.spatial_embed = SpatialEncoding(embed_dim, dropout=dropout,
                                                  lat_mean=lat_mean, lon_mean=lon_mean,
                                                  lat_std=lat_std, lon_std=lon_std)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, poi_ids: torch.Tensor, timestamps: Optional[torch.Tensor] = None,
                locations: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = self.poi_embed(poi_ids)
        x = self.pos_embed(x)
        if self.use_temporal and timestamps is not None:
            x = x + self.temporal_embed(timestamps)
        if self.use_spatial and locations is not None:
            x = x + self.spatial_embed(locations)
        return self.layer_norm(x)

print("Embedding layers defined.")

Embedding layers defined.


In [7]:
# Section 7: Candidate-Relative Biases (KEY INNOVATION - from biases.py)

class TemporalBias(nn.Module):
    """
    Candidate-relative temporal bias: b_t(Delta_t_i)
    Paper: "logarithmically-spaced temporal buckets with boundaries at
    1 hour, 6 hours, 24 hours, 7 days, and 30 days, creating 6 distinct time ranges"
    """
    # 6 buckets: [0,1h), [1h,6h), [6h,24h), [24h,7d), [7d,30d), [30d,inf)
    TIME_BUCKETS = [0, 3600, 6*3600, 24*3600, 7*24*3600, 30*24*3600, float('inf')]

    def __init__(self, num_buckets: int = 6, hidden_dim: int = 64):
        super().__init__()
        self.num_buckets = num_buckets
        self.bucket_embedding = nn.Embedding(num_buckets, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        nn.init.normal_(self.bucket_embedding.weight, std=0.02)
        nn.init.normal_(self.output.weight, std=0.02)
        nn.init.zeros_(self.output.bias)

    def _get_time_bucket(self, time_gap: torch.Tensor) -> torch.Tensor:
        buckets = torch.zeros_like(time_gap, dtype=torch.long)
        for i, (lower, upper) in enumerate(zip(self.TIME_BUCKETS[:-1], self.TIME_BUCKETS[1:])):
            buckets[(time_gap >= lower) & (time_gap < upper)] = i
        return buckets.clamp(0, self.num_buckets - 1)

    def forward(self, time_gaps: torch.Tensor) -> torch.Tensor:
        bucket_emb = self.bucket_embedding(self._get_time_bucket(time_gaps))
        return self.output(bucket_emb).squeeze(-1)


class SpatialBias(nn.Module):
    """
    Candidate-relative spatial bias - THE KEY INNOVATION!
    b_s(Delta_d_i(c)) where Delta_d_i(c) = distance(l_i, c)

    Paper: "spatial buckets with boundaries at 100 meters, 500 meters,
    2 kilometers, and 10 kilometers" -> 5 buckets
    """
    # Default 5 buckets: [0,0.1km), [0.1,0.5km), [0.5,2km), [2,10km), [10km,inf)
    DEFAULT_DIST_BUCKETS = [0, 0.1, 0.5, 2, 10, float('inf')]

    def __init__(self, hidden_dim: int = 64, dist_buckets: list = None):
        super().__init__()
        self.DIST_BUCKETS = dist_buckets if dist_buckets is not None else self.DEFAULT_DIST_BUCKETS
        self.num_buckets = len(self.DIST_BUCKETS) - 1
        self.bucket_embedding = nn.Embedding(self.num_buckets, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        nn.init.normal_(self.bucket_embedding.weight, std=0.02)
        nn.init.normal_(self.output.weight, std=0.02)
        nn.init.zeros_(self.output.bias)

    @staticmethod
    def haversine_distance(loc1: torch.Tensor, loc2: torch.Tensor) -> torch.Tensor:
        """Compute haversine distance between coordinates. Returns distances in km."""
        R = 6371.0  # Earth radius in km
        lat1, lon1 = torch.deg2rad(loc1[..., 0]), torch.deg2rad(loc1[..., 1])
        lat2, lon2 = torch.deg2rad(loc2[..., 0]), torch.deg2rad(loc2[..., 1])
        dlat, dlon = lat2 - lat1, lon2 - lon1

        a = torch.sin(dlat/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin(dlon/2)**2
        a = torch.clamp(a, min=0.0, max=1.0)
        c = 2 * torch.asin(torch.sqrt(a + 1e-10))
        return R * c

    def _get_dist_bucket(self, distances: torch.Tensor) -> torch.Tensor:
        buckets = torch.zeros_like(distances, dtype=torch.long)
        for i, (lower, upper) in enumerate(zip(self.DIST_BUCKETS[:-1], self.DIST_BUCKETS[1:])):
            buckets[(distances >= lower) & (distances < upper)] = i
        return buckets.clamp(0, self.num_buckets - 1)

    def forward(self, history_locations: torch.Tensor, candidate_locations: torch.Tensor) -> torch.Tensor:
        """Returns [B, L, C] spatial bias values (DIFFERENT for each candidate!)"""
        hist_exp = history_locations.unsqueeze(2)  # [B, L, 1, 2]
        cand_exp = candidate_locations.unsqueeze(1)  # [B, 1, C, 2]

        distances = self.haversine_distance(hist_exp, cand_exp)  # [B, L, C]

        bucket_emb = self.bucket_embedding(self._get_dist_bucket(distances))
        return self.output(bucket_emb).squeeze(-1)


class CombinedBias(nn.Module):
    """Combined temporal and spatial bias: b_t(Delta_t) + b_s(Delta_d(c))"""
    def __init__(self, use_temporal: bool = True, use_spatial: bool = True, hidden_dim: int = 64, dist_buckets: list = None):
        super().__init__()
        self.use_temporal, self.use_spatial = use_temporal, use_spatial
        if use_temporal:
            self.temporal_bias = TemporalBias(num_buckets=6, hidden_dim=hidden_dim)
        if use_spatial:
            self.spatial_bias = SpatialBias(hidden_dim=hidden_dim, dist_buckets=dist_buckets)

    def forward(self, time_gaps: Optional[torch.Tensor] = None,
                history_locations: Optional[torch.Tensor] = None,
                candidate_locations: Optional[torch.Tensor] = None) -> torch.Tensor:
        B = time_gaps.shape[0] if time_gaps is not None else history_locations.shape[0]
        L = time_gaps.shape[1] if time_gaps is not None else history_locations.shape[1]
        C = candidate_locations.shape[1] if candidate_locations is not None else 1
        dev = time_gaps.device if time_gaps is not None else history_locations.device

        total_bias = torch.zeros(B, L, C, device=dev)
        if self.use_temporal and time_gaps is not None:
            # temporal bias: [B, L] -> [B, L, 1] broadcast over C candidates
            total_bias = total_bias + self.temporal_bias(time_gaps).unsqueeze(-1)
        if self.use_spatial and history_locations is not None and candidate_locations is not None:
            total_bias = total_bias + self.spatial_bias(history_locations, candidate_locations)

        return total_bias

print("Bias modules defined.")

Bias modules defined.


In [8]:
# Section 8: Attention Mechanisms (CORE INNOVATION - from attention.py)

class CandidateConditionedAttention(nn.Module):
    """
    Multi-head candidate-conditioned cross-attention.
    CORE INNOVATION: Candidate POIs serve as queries, historical visits serve as keys/values.
    Formula: alpha_i^(c) propto exp(q_c^T k_i / sqrt(d) + b_t(Delta_t_i) + b_s(Delta_d_i(c)))
    """
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1,
                 max_attn_score: float = 50.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = math.sqrt(self.head_dim)
        self.max_attn_score = max_attn_score

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        for proj in [self.q_proj, self.k_proj, self.v_proj, self.out_proj]:
            nn.init.xavier_uniform_(proj.weight)
            nn.init.zeros_(proj.bias)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                bias: Optional[torch.Tensor] = None,
                key_padding_mask: Optional[torch.Tensor] = None,
                return_attention: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, C, D = query.shape
        L = key.shape[1]
        H = self.num_heads

        Q = self.q_proj(query).view(B, C, H, self.head_dim).transpose(1, 2)
        K = self.k_proj(key).view(B, L, H, self.head_dim).transpose(1, 2)
        V = self.v_proj(value).view(B, L, H, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Add candidate-relative bias (KEY INNOVATION!)
        if bias is not None:
            # bias: [B, L, C] -> permute to [B, C, L] -> [B, 1, C, L] for broadcast over heads
            attn_scores = attn_scores + bias.permute(0, 2, 1).unsqueeze(1)

        # Clamp attention scores to prevent softmax overflow
        attn_scores = torch.clamp(attn_scores, min=-self.max_attn_score, max=self.max_attn_score)

        # Apply key padding mask: [B, L] -> [B, 1, 1, L]
        if key_padding_mask is not None:
            attn_scores = attn_scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e4)

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, C, D)
        output = self.out_proj(output)

        return (output, attn_weights) if return_attention else (output, None)


class CandidateConditionedEncoder(nn.Module):
    """Full encoder block with candidate-conditioned attention + FFN."""
    def __init__(self, embed_dim: int, num_heads: int = 8, ff_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.attention = CandidateConditionedAttention(embed_dim, num_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim), nn.Dropout(dropout)
        )
        self.norm1, self.norm2 = nn.LayerNorm(embed_dim), nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, bias=None, key_padding_mask=None, return_attention=False):
        attn_out, attn_weights = self.attention(
            self.norm1(query), self.norm1(key), self.norm1(value),
            bias=bias, key_padding_mask=key_padding_mask, return_attention=return_attention
        )
        query = query + self.dropout(attn_out)
        query = query + self.ff(self.norm2(query))
        return query, attn_weights


class HistorySelfAttention(nn.Module):
    """
    Self-attention over user history (before candidate conditioning).

    CRITICAL: History is LEFT-PADDED. With a causal mask, left-padded positions
    have ALL causal targets also masked, producing softmax(all -inf) = NaN.
    We implement manual attention to handle this correctly by using a large
    negative value (-1e4) instead of -inf so softmax produces near-zero
    (not NaN), and then zero out padded positions in the output.
    """
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert embed_dim % num_heads == 0

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout)
        )
        self.norm1, self.norm2 = nn.LayerNorm(embed_dim), nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, L, D = x.shape
        H = self.num_heads

        x_norm = self.norm1(x)

        Q = self.q_proj(x_norm).view(B, L, H, self.head_dim).transpose(1, 2)  # [B, H, L, d]
        K = self.k_proj(x_norm).view(B, L, H, self.head_dim).transpose(1, 2)
        V = self.v_proj(x_norm).view(B, L, H, self.head_dim).transpose(1, 2)

        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # [B, H, L, L]

        # Causal mask: prevent attending to future positions
        causal_mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), -1e4)

        # Padding mask: prevent attending to padded positions
        # key_padding_mask: [B, L], True = padded
        if key_padding_mask is not None:
            attn_scores = attn_scores.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e4)

        # Using -1e4 instead of -inf means softmax produces near-zero (not NaN)
        # even when all positions are masked for left-padded rows.
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_out = torch.matmul(attn_weights, V)  # [B, H, L, d]
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, L, D)
        attn_out = self.out_proj(attn_out)

        # Zero out attention output at padded positions
        if key_padding_mask is not None:
            attn_out = attn_out.masked_fill(key_padding_mask.unsqueeze(-1), 0.0)

        x = x + attn_out

        ff_out = self.ff(self.norm2(x))
        if key_padding_mask is not None:
            ff_out = ff_out.masked_fill(key_padding_mask.unsqueeze(-1), 0.0)
        x = x + ff_out

        return x

print("Attention mechanisms defined.")

Attention mechanisms defined.


In [9]:
# Section 9: Main CaST-POI Model (from cast_poi.py)

class PredictionHead(nn.Module):
    """Prediction head: MLP([h_u^(c); e_c; h_u^(c) * e_c])"""
    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, 1)
        )

    def forward(self, user_repr: torch.Tensor, candidate_emb: torch.Tensor) -> torch.Tensor:
        combined = torch.cat([user_repr, candidate_emb, user_repr * candidate_emb], dim=-1)
        return self.mlp(combined).squeeze(-1)


class CaSTPOI(nn.Module):
    """
    CaST-POI: Candidate-Aware Spatiotemporal POI Recommendation Model.
    Key Innovation: Candidate-specific user representations via cross-attention.
    """
    def __init__(self, num_pois: int, embed_dim: int = 64, num_heads: int = 8,
                 num_layers: int = 2, ff_dim: int = 256, max_seq_len: int = 100,
                 dropout: float = 0.1, use_temporal_bias: bool = True,
                 use_spatial_bias: bool = True, use_history_self_attn: bool = True,
                 lat_mean: float = 40.0, lon_mean: float = -74.0,
                 lat_std: float = 1.0, lon_std: float = 1.0,
                 dist_buckets: list = None):
        super().__init__()
        self.num_pois, self.embed_dim = num_pois, embed_dim
        self.use_temporal_bias = use_temporal_bias
        self.use_spatial_bias = use_spatial_bias
        self.use_history_self_attn = use_history_self_attn

        self.history_embed = CombinedEmbedding(num_pois, embed_dim, max_seq_len, True, True, dropout,
                                                lat_mean=lat_mean, lon_mean=lon_mean,
                                                lat_std=lat_std, lon_std=lon_std)
        self.candidate_embed = POIEmbedding(num_pois, embed_dim, dropout=dropout)

        if use_temporal_bias or use_spatial_bias:
            self.bias_module = CombinedBias(use_temporal_bias, use_spatial_bias, embed_dim, dist_buckets=dist_buckets)
        else:
            self.bias_module = None

        if use_history_self_attn:
            self.history_self_attn = HistorySelfAttention(embed_dim, num_heads, dropout)
        else:
            self.history_self_attn = None

        self.encoder_layers = nn.ModuleList([
            CandidateConditionedEncoder(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.prediction_head = PredictionHead(embed_dim, dropout)

    def compute_bias(self, timestamps, history_locations, candidate_locations):
        if self.bias_module is None:
            return None
        time_gaps = torch.clamp(timestamps[:, -1:] - timestamps, min=0)
        return self.bias_module(time_gaps, history_locations, candidate_locations)

    def forward(self, history_pois, history_timestamps, history_locations, history_len,
                candidates, candidate_locations, return_attention=False):
        B, L = history_pois.shape
        C = candidates.shape[1]

        # Padding mask
        positions = torch.arange(L, device=history_pois.device).unsqueeze(0)
        key_padding_mask = positions < (L - history_len.unsqueeze(1))

        # Embed history and candidates
        history_emb = self.history_embed(history_pois, history_timestamps, history_locations)
        if self.history_self_attn is not None:
            history_emb = self.history_self_attn(history_emb, key_padding_mask)

        candidate_emb = self.candidate_embed(candidates)
        bias = self.compute_bias(history_timestamps, history_locations, candidate_locations)

        # Candidate-conditioned encoding
        user_repr, attention_weights = candidate_emb, None
        for encoder in self.encoder_layers:
            user_repr, attn = encoder(user_repr, history_emb, history_emb, bias, key_padding_mask, return_attention)
            if return_attention:
                attention_weights = attn

        scores = self.prediction_head(user_repr, candidate_emb)
        result = {'scores': scores, 'user_repr': user_repr}
        if return_attention:
            result['attention_weights'] = attention_weights
        return result


class CaSTPOILoss(nn.Module):
    """Cross-entropy loss with optional explore reweighting."""
    def __init__(self, label_smoothing: float = 0.0, explore_weight: float = 1.0):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing, reduction='none')
        self.explore_weight = explore_weight

    def forward(self, scores: torch.Tensor, target_idx: torch.Tensor, is_explore: torch.Tensor = None) -> torch.Tensor:
        # scores: [B, C], target_idx: [B]
        per = self.loss_fn(scores, target_idx)  # [B]
        if is_explore is None:
            return per.mean()
        w = torch.ones_like(per)
        w[is_explore] = self.explore_weight
        return (per * w).mean()


# Test model
print("Testing CaST-POI model...")
test_model = CaSTPOI(num_pois=1000, embed_dim=64, num_heads=8, num_layers=2)
num_params = sum(p.numel() for p in test_model.parameters())
print(f"Model parameters: {num_params:,}")
del test_model

Testing CaST-POI model...
Model parameters: 297,667


---
## Part 4: Dataset & DataLoader

In [10]:
# Section 10: POIDataset with Full-POI Evaluation Support

class POIDataset(Dataset):
    """
    POI Dataset supporting:
    - Popularity-based hard negative sampling for training
    - Full POI candidate evaluation (candidate_pool_size=None) to match paper protocol
    - Configurable candidate pool size (for Figure 3 experiment)
    - Train/test modes with proper history construction
    """
    def __init__(self, data: Dict, mode: str = 'train', max_history_len: int = 50,
                 num_negatives: int = 99, candidate_pool_size: Optional[int] = None):
        self.mode = mode
        self.max_history_len = max_history_len
        self.num_negatives = num_negatives
        self.num_pois = data['stats']['num_pois']

        # candidate_pool_size=None means full POI evaluation
        # For training, always use num_negatives+1
        if mode == 'train':
            self.candidate_pool_size = num_negatives + 1
        else:
            self.candidate_pool_size = candidate_pool_size  # None = all POIs

        # Build POI location lookup
        poi_info = data['poi_info']
        self.poi_locations = np.zeros((self.num_pois, 2))
        for _, row in poi_info.iterrows():
            idx = int(row['poi_idx'])
            if idx < self.num_pois:
                self.poi_locations[idx] = [row['latitude'], row['longitude']]

        # Fill missing POI coordinates with dataset centroid
        valid_locs = self.poi_locations[(self.poi_locations[:, 0] != 0) | (self.poi_locations[:, 1] != 0)]
        if len(valid_locs) > 0:
            self.default_lat = float(np.mean(valid_locs[:, 0]))
            self.default_lon = float(np.mean(valid_locs[:, 1]))
        else:
            self.default_lat = 40.7
            self.default_lon = -74.0

        zero_mask = (self.poi_locations[:, 0] == 0) & (self.poi_locations[:, 1] == 0)
        self.poi_locations[zero_mask] = [self.default_lat, self.default_lon]

        # Build POI popularity distribution for hard negative sampling
        self._build_popularity_distribution(data)

        self.samples = self._create_samples(data)
        eval_mode_str = f"full POI ({self.num_pois})" if self.candidate_pool_size is None else f"pool={self.candidate_pool_size}"
        print(f"[{mode}] {len(self.samples)} samples, {(~zero_mask).sum()} valid POI locations, eval: {eval_mode_str}")

    def _build_popularity_distribution(self, data: Dict):
        """Build popularity-based sampling distribution over POIs (indices 1..num_pois-1)."""
        poi_counts = np.zeros(self.num_pois, dtype=np.float64)
        for traj in data['train_data'].values():
            for visit in traj:
                poi_counts[visit['poi_idx']] += 1
        poi_counts[0] = 0
        total = poi_counts.sum()
        if total > 0:
            self.poi_popularity = poi_counts / total
        else:
            self.poi_popularity = np.ones(self.num_pois) / (self.num_pois - 1)
            self.poi_popularity[0] = 0
        self.valid_poi_indices = np.where(self.poi_popularity > 0)[0]

    def _create_samples(self, data: Dict) -> List[Dict]:
        samples = []
        train_data, test_data = data['train_data'], data['test_data']

        if self.mode == 'train':
            for user_idx, traj in train_data.items():
                for i in range(1, len(traj)):
                    samples.append({'user_idx': user_idx, 'history': traj[:i], 'target': traj[i]})
        else:
            for user_idx, traj in test_data.items():
                train_traj = train_data.get(user_idx, [])
                if not train_traj:
                    continue
                for i, target in enumerate(traj):
                    history = train_traj + traj[:i] if i > 0 else train_traj
                    samples.append({'user_idx': user_idx, 'history': history, 'target': target})
        return samples

    def _sample_negatives(self, target_poi: int, num_neg: int) -> List[int]:
        """Sample negative POIs using popularity-based hard negative strategy."""
        negatives = set()
        while len(negatives) < num_neg:
            candidates = np.random.choice(
                self.valid_poi_indices,
                size=min(num_neg * 2, len(self.valid_poi_indices)),
                replace=False,
                p=self.poi_popularity[self.valid_poi_indices] / self.poi_popularity[self.valid_poi_indices].sum()
            )
            for c in candidates:
                if c != target_poi and c != 0 and c not in negatives:
                    negatives.add(int(c))
                    if len(negatives) >= num_neg:
                        break
        return list(negatives)[:num_neg]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.samples[idx]
        history = sample['history'][-self.max_history_len:]

        poi_indices = [h['poi_idx'] for h in history]
        timestamps = [h['timestamp'] for h in history]
        locations = [(h['latitude'], h['longitude']) for h in history]

        target = sample['target']
        target_poi = target['poi_idx']

        if self.mode == 'train':
            # Training: 1 positive + num_negatives negatives, target at position 0
            negatives = self._sample_negatives(target_poi, self.num_negatives)
            candidates = [target_poi] + negatives
            target_idx = 0
        elif self.candidate_pool_size is None:
            # Full POI evaluation: all POIs as candidates, target_idx = target_poi index
            candidates = list(range(self.num_pois))
            target_idx = target_poi
        else:
            # Sampled pool evaluation (for Figure 3 experiment)
            num_neg = self.candidate_pool_size - 1
            negatives = self._sample_negatives(target_poi, num_neg)
            target_position = np.random.randint(0, self.candidate_pool_size)
            candidates = negatives[:target_position] + [target_poi] + negatives[target_position:]
            target_idx = target_position

        candidate_locations = [self.poi_locations[c].tolist() for c in candidates]

        # Pad history (left-padding)
        history_len = len(poi_indices)
        pad_len = self.max_history_len - history_len
        if pad_len > 0:
            ref_timestamp = timestamps[0] if timestamps else 0
            ref_location = (self.default_lat, self.default_lon)
            poi_indices = [0] * pad_len + poi_indices
            timestamps = [ref_timestamp] * pad_len + timestamps
            locations = [ref_location] * pad_len + locations

        return {
            'user_idx': torch.tensor(sample['user_idx'], dtype=torch.long),
            'history_pois': torch.tensor(poi_indices, dtype=torch.long),
            'history_timestamps': torch.tensor(timestamps, dtype=torch.float32),
            'history_locations': torch.tensor(locations, dtype=torch.float32),
            'history_len': torch.tensor(history_len, dtype=torch.long),
            'candidates': torch.tensor(candidates, dtype=torch.long),
            'candidate_locations': torch.tensor(candidate_locations, dtype=torch.float32),
            'target_idx': torch.tensor(target_idx, dtype=torch.long),
            'target_poi': torch.tensor(target_poi, dtype=torch.long)
        }


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Custom collate function that handles variable-length candidates."""
    result = {}
    for key in batch[0].keys():
        if key in ('candidates', 'candidate_locations'):
            max_len = max(b[key].shape[0] for b in batch)
            padded = []
            for b in batch:
                pad_size = max_len - b[key].shape[0]
                if pad_size > 0:
                    if key == 'candidates':
                        padded.append(torch.cat([b[key], torch.zeros(pad_size, dtype=torch.long)]))
                    else:
                        padded.append(torch.cat([b[key], torch.zeros(pad_size, 2)]))
                else:
                    padded.append(b[key])
            result[key] = torch.stack(padded)
        else:
            result[key] = torch.stack([b[key] for b in batch])
    return result


def get_dataloader(data: Dict, mode: str, batch_size: int, max_history_len: int = 50,
                   num_negatives: int = 99, candidate_pool_size: Optional[int] = None,
                   num_workers: int = 0) -> DataLoader:
    dataset = POIDataset(data, mode, max_history_len, num_negatives, candidate_pool_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=(mode == 'train'),
                      collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)


# Create dataloaders
# Training: use sampled negatives (100 candidates) for efficiency
# Evaluation: use ALL POIs as candidates to match paper protocol (Table 1)
print("\nCreating dataloaders...")
train_loader = get_dataloader(data, 'train', CONFIG['batch_size'], CONFIG['max_history_len'], CONFIG['num_negatives'])
test_loader = get_dataloader(data, 'test', CONFIG['eval_batch_size'], CONFIG['max_history_len'],
                              CONFIG['num_negatives'], candidate_pool_size=None)  # Full POI evaluation
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")
print(f"Evaluation: full POI ranking ({data['stats']['num_pois']} candidates per sample)")


Creating dataloaders...
[train] 47772 samples, 2590 valid POI locations, eval: pool=100
[test] 14940 samples, 2590 valid POI locations, eval: full POI (2591)
Train batches: 747, Test batches: 934
Evaluation: full POI ranking (2591 candidates per sample)


---
## Part 5: Evaluation Metrics

In [11]:
# Section 11: Metrics (from metrics.py)

def hit_rate_at_k(scores: torch.Tensor, target_idx: torch.Tensor, k: int) -> float:
    _, top_k = scores.topk(k, dim=1)
    return (top_k == target_idx.unsqueeze(1)).any(dim=1).float().mean().item()

def ndcg_at_k(scores: torch.Tensor, target_idx: torch.Tensor, k: int) -> float:
    B = scores.shape[0]
    sorted_indices = scores.argsort(dim=1, descending=True)
    ranks = torch.zeros(B, device=scores.device)
    for i in range(B):
        rank = (sorted_indices[i] == target_idx[i]).nonzero(as_tuple=True)[0]
        if len(rank) > 0:
            ranks[i] = rank[0].item() + 1
    dcg = torch.zeros(B, device=scores.device)
    mask = (ranks > 0) & (ranks <= k)
    dcg[mask] = 1.0 / torch.log2(ranks[mask] + 1)
    return dcg.mean().item()

def mrr(scores: torch.Tensor, target_idx: torch.Tensor) -> float:
    B = scores.shape[0]
    sorted_indices = scores.argsort(dim=1, descending=True)
    rr = torch.zeros(B, device=scores.device)
    for i in range(B):
        rank = (sorted_indices[i] == target_idx[i]).nonzero(as_tuple=True)[0]
        if len(rank) > 0:
            rr[i] = 1.0 / (rank[0].item() + 1)
    return rr.mean().item()

def compute_all_metrics(scores: torch.Tensor, target_idx: torch.Tensor, ks: List[int] = [5, 10, 20]) -> Dict[str, float]:
    metrics = {}
    for k in ks:
        metrics[f'HR@{k}'] = hit_rate_at_k(scores, target_idx, k)
        metrics[f'NDCG@{k}'] = ndcg_at_k(scores, target_idx, k)
    metrics['MRR'] = mrr(scores, target_idx)
    return metrics

class MetricsAccumulator:
    def __init__(self, ks: List[int] = [5, 10, 20]):
        self.ks = ks
        self.reset()

    def reset(self):
        self.total_samples = 0
        self.metrics_sum = {f'HR@{k}': 0.0 for k in self.ks}
        self.metrics_sum.update({f'NDCG@{k}': 0.0 for k in self.ks})
        self.metrics_sum['MRR'] = 0.0

    def update(self, scores: torch.Tensor, target_idx: torch.Tensor):
        B = scores.shape[0]
        self.total_samples += B
        batch_metrics = compute_all_metrics(scores, target_idx, self.ks)
        for key, value in batch_metrics.items():
            self.metrics_sum[key] += value * B

    def compute(self) -> Dict[str, float]:
        if self.total_samples == 0:
            return {k: 0.0 for k in self.metrics_sum.keys()}
        return {key: value / self.total_samples for key, value in self.metrics_sum.items()}

def format_metrics(metrics: Dict[str, float], as_percentage: bool = True) -> str:
    lines = []
    for key in sorted(metrics.keys()):
        value = metrics[key]
        lines.append(f"{key}: {value * 100:.2f}%" if as_percentage else f"{key}: {value:.4f}")
    return " | ".join(lines)

print("Metrics functions defined.")

Metrics functions defined.


In [ ]:
# Section 11.5: Results Saving Utilities

import json
from datetime import datetime

def save_experiment_results(dataset_name, results_dict, filename=None):
    """Save experiment results to JSON file"""
    if filename is None:
        filename = f'results_{dataset_name}_{datetime.now():%Y%m%d_%H%M}.json'
    
    def convert(obj):
        """Convert numpy/torch types to Python native types"""
        if hasattr(obj, 'item'):
            return obj.item()
        elif isinstance(obj, dict):
            return {k: convert(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert(v) for v in obj]
        elif isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return obj
    
    with open(filename, 'w') as f:
        json.dump(convert(results_dict), f, indent=2)
    print(f"Results saved to {filename}")
    return filename

def print_results_table(main_metrics, ablation_results=None, dataset_name=''):
    """Print formatted results table"""
    print(f"\n{'='*70}")
    print(f"RESULTS FOR {dataset_name.upper()}")
    print('='*70)
    
    # Main results
    print(f"\n[Main Model Results]")
    print(f"HR@5: {main_metrics['HR@5']*100:.2f}%  |  HR@10: {main_metrics['HR@10']*100:.2f}%  |  HR@20: {main_metrics['HR@20']*100:.2f}%")
    print(f"NDCG@5: {main_metrics['NDCG@5']*100:.2f}%  |  NDCG@10: {main_metrics['NDCG@10']*100:.2f}%  |  NDCG@20: {main_metrics['NDCG@20']*100:.2f}%")
    print(f"MRR: {main_metrics['MRR']*100:.2f}%")
    
    # Ablation results
    if ablation_results:
        print(f"\n[Ablation Study]")
        print(f"{'Configuration':<25} {'HR@10':>10} {'NDCG@10':>10} {'MRR':>10}")
        print('-'*55)
        for result in ablation_results:
            m = result['metrics']
            print(f"{result['name']:<25} {m['HR@10']*100:>9.2f}% {m['NDCG@10']*100:>9.2f}% {m['MRR']*100:>9.2f}%")

print("Results saving utilities defined.")

---
## Part 6: Training & Evaluation

In [12]:
# Section 12 & 13: Training Loop with Warmup + Cosine Schedule

def save_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics
    }, path)
    print(f"Checkpoint saved to {path}")

def load_checkpoint(model, path, device, optimizer=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Loaded checkpoint from {path} (epoch {checkpoint['epoch']})")
    return checkpoint


class WarmupCosineScheduler:
    """Linear warmup then cosine decay, matching paper's training protocol."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, eta_min=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.eta_min = eta_min
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        if self.current_epoch <= self.warmup_epochs:
            # Linear warmup
            factor = self.current_epoch / max(self.warmup_epochs, 1)
        else:
            # Cosine decay
            progress = (self.current_epoch - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
            factor = 0.5 * (1 + math.cos(math.pi * progress))

        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = self.eta_min + (base_lr - self.eta_min) * factor


def train_epoch(model, dataloader, optimizer, criterion, device, gradient_clip=5.0):
    model.train()
    total_loss, num_batches = 0.0, 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        output = model(
            history_pois=batch['history_pois'],
            history_timestamps=batch['history_timestamps'],
            history_locations=batch['history_locations'],
            history_len=batch['history_len'],
            candidates=batch['candidates'],
            candidate_locations=batch['candidate_locations']
        )

        # -------- identify explore samples --------
        B = batch['history_pois'].size(0)
        is_explore = torch.zeros(B, dtype=torch.bool, device=device)
        for i in range(B):
            L = int(batch['history_len'][i].item())
            hist = batch['history_pois'][i, :L]
            is_explore[i] = ~(hist == batch['target_poi'][i]).any()

        loss = criterion(
            output['scores'],
            batch['target_idx'],
            is_explore=is_explore
        )

        # Safety check
        if torch.isnan(loss) or torch.isinf(loss):
            continue

        loss.backward()

        if gradient_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)

        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate(model, dataloader, device, ks=[5, 10, 20]):
    model.eval()
    accumulator = MetricsAccumulator(ks=ks)

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        output = model(
            history_pois=batch['history_pois'],
            history_timestamps=batch['history_timestamps'],
            history_locations=batch['history_locations'],
            history_len=batch['history_len'],
            candidates=batch['candidates'],
            candidate_locations=batch['candidate_locations']
        )

        accumulator.update(output['scores'], batch['target_idx'])

    return accumulator.compute()

def train_model(model, train_loader, test_loader, config, device, save_path=None, verbose=True):
    model = model.to(device)
    criterion = CaSTPOILoss(config.get('label_smoothing', 0.0), config.get('explore_weight', 1.0))
    optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])

    warmup_epochs = config.get('warmup_epochs', 3)
    scheduler = WarmupCosineScheduler(optimizer, warmup_epochs, config['num_epochs'], eta_min=1e-6)

    history = {'train_loss': [], 'metrics': []}
    best_metric, best_state, patience_counter = 0, None, 0

    for epoch in range(1, config['num_epochs'] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, config['gradient_clip'])
        scheduler.step()
        history['train_loss'].append(train_loss)

        metrics = evaluate(model, test_loader, device, config['eval_ks'])
        history['metrics'].append(metrics)

        current_lr = optimizer.param_groups[0]['lr']
        if verbose:
            print(f"Epoch {epoch:3d} | Loss: {train_loss:.4f} | LR: {current_lr:.6f} | {format_metrics(metrics)}")

        current_metric = metrics['HR@10']
        if current_metric > best_metric:
            best_metric = current_metric
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            if save_path:
                save_checkpoint(model, optimizer, epoch, metrics, save_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stopping_patience']:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break

    if best_state:
        model.load_state_dict(best_state)

    return model, history, best_metric

print("Training functions defined.")

Training functions defined.


---
## Part 7: Experiments

In [13]:
# Section 14: Main Experiment - Train Full Model

print("\n" + "="*80)
print("MAIN EXPERIMENT: Training Full CaST-POI Model")
print("="*80)

# Compute spatial normalization statistics from actual data
poi_lats = data['poi_info']['latitude'].values
poi_lons = data['poi_info']['longitude'].values
lat_mean = float(np.mean(poi_lats))
lon_mean = float(np.mean(poi_lons))
lat_std = float(np.std(poi_lats)) + 1e-6
lon_std = float(np.std(poi_lons)) + 1e-6
print(f"Spatial normalization: lat={lat_mean:.4f}±{lat_std:.4f}, lon={lon_mean:.4f}±{lon_std:.4f}")

# Create model
model = CaSTPOI(
    num_pois=data['stats']['num_pois'],
    embed_dim=CONFIG['embed_dim'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    ff_dim=CONFIG['ff_dim'],
    max_seq_len=CONFIG['max_seq_len'],
    dropout=CONFIG['dropout'],
    use_temporal_bias=True,
    use_spatial_bias=True,
    use_history_self_attn=True,
    lat_mean=lat_mean, lon_mean=lon_mean,
    lat_std=lat_std, lon_std=lon_std,
    dist_buckets=CONFIG.get('dist_buckets', None)
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Train
model, history, best_hr10 = train_model(
    model, train_loader, test_loader, CONFIG, device,
    save_path='best_model.pt'
)

# Final evaluation
final_metrics = evaluate(model, test_loader, device, CONFIG['eval_ks'])
print(f"\nFinal Results: {format_metrics(final_metrics)}")
print(f"Best HR@10: {best_hr10*100:.2f}%")
# --- Extra evaluation for general next-POI: revisit vs explore vs balanced ---
from torch.utils.data import DataLoader, Subset
import random

def split_indices_by_revisit(dataset):
    revisit_idx, explore_idx = [], []
    for i, s in enumerate(dataset.samples):
        hist = {h['poi_idx'] for h in s['history']}
        tgt = s['target']['poi_idx']
        (revisit_idx if tgt in hist else explore_idx).append(i)
    return revisit_idx, explore_idx

def make_subset_loader(base_loader, indices):
    ds = base_loader.dataset
    return DataLoader(
        Subset(ds, indices),
        batch_size=base_loader.batch_size,
        shuffle=False,
        num_workers=getattr(base_loader, "num_workers", 0),
        collate_fn=base_loader.collate_fn
    )

revisit_idx, explore_idx = split_indices_by_revisit(test_loader.dataset)
print(f"\n[test split] revisit={len(revisit_idx)}, explore={len(explore_idx)}, explore%={len(explore_idx)/len(test_loader.dataset)*100:.2f}%")

test_revisit_loader = make_subset_loader(test_loader, revisit_idx)
test_explore_loader = make_subset_loader(test_loader, explore_idx)

m = min(len(revisit_idx), len(explore_idx))
balanced_idx = random.sample(revisit_idx, m) + random.sample(explore_idx, m)
random.shuffle(balanced_idx)
test_balanced_loader = make_subset_loader(test_loader, balanced_idx)

metrics_revisit  = evaluate(model, test_revisit_loader, device, CONFIG['eval_ks'])
metrics_explore  = evaluate(model, test_explore_loader, device, CONFIG['eval_ks'])
metrics_balanced = evaluate(model, test_balanced_loader, device, CONFIG['eval_ks'])

print("Revisit  :", format_metrics(metrics_revisit))
print("Explore  :", format_metrics(metrics_explore))
print("Balanced :", format_metrics(metrics_balanced))



MAIN EXPERIMENT: Training Full CaST-POI Model
Spatial normalization: lat=40.7541±0.0753, lon=-73.9741±0.0869
Model parameters: 501,315


Training:   0%|          | 0/747 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

Epoch   1 | Loss: 5.3142 | LR: 0.000334 | HR@10: 51.37% | HR@20: 61.45% | HR@5: 38.76% | MRR: 25.87% | NDCG@10: 30.96% | NDCG@20: 33.51% | NDCG@5: 26.85%
Checkpoint saved to best_model.pt


Training:   0%|          | 0/747 [00:00<?, ?it/s]

...


Final Results: HR@10: 67.09% | HR@20: 73.24% | HR@5: 56.09% | MRR: 35.98% | NDCG@10: 42.89% | NDCG@20: 44.46% | NDCG@5: 39.29%
Best HR@10: 67.09%

[test split] revisit=12494, explore=2446, explore%=16.37%


Evaluating:   0%|          | 0/781 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/153 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/306 [00:00<?, ?it/s]

Revisit  : HR@10: 79.35% | HR@20: 86.08% | HR@5: 66.57% | MRR: 42.60% | NDCG@10: 50.89% | NDCG@20: 52.61% | NDCG@5: 46.70%
Explore  : HR@10: 4.46% | HR@20: 7.65% | HR@5: 2.58% | MRR: 2.17% | NDCG@10: 2.05% | NDCG@20: 2.85% | NDCG@5: 1.44%
Balanced : HR@10: 41.29% | HR@20: 46.30% | HR@5: 33.50% | MRR: 22.03% | NDCG@10: 26.03% | NDCG@20: 27.31% | NDCG@5: 23.48%


In [17]:
# Section 15: Ablation Study (Table 3)

ABLATION_CONFIGS = [
    {"name": "Full Model", "use_temporal_bias": True, "use_spatial_bias": True, "use_history_self_attn": True},
    {"name": "w/o Temporal Bias", "use_temporal_bias": False, "use_spatial_bias": True, "use_history_self_attn": True},
    {"name": "w/o Spatial Bias", "use_temporal_bias": True, "use_spatial_bias": False, "use_history_self_attn": True},
    {"name": "w/o Both Biases", "use_temporal_bias": False, "use_spatial_bias": False, "use_history_self_attn": True},
    {"name": "w/o History Self-Attn", "use_temporal_bias": True, "use_spatial_bias": True, "use_history_self_attn": False},
]

def run_ablation_study(configs, data, config, device, num_epochs=15):
    results = []
    ablation_config = config.copy()
    ablation_config['num_epochs'] = num_epochs
    ablation_config['early_stopping_patience'] = 3

    for cfg in configs:
        print(f"\n{'='*60}")
        print(f"Ablation: {cfg['name']}")
        print("="*60)

        set_seed(42)
        ablation_model = CaSTPOI(
            num_pois=data['stats']['num_pois'],
            embed_dim=config['embed_dim'],
            num_heads=config['num_heads'],
            num_layers=config['num_layers'],
            ff_dim=config['ff_dim'],
            max_seq_len=config['max_seq_len'],
            dropout=config['dropout'],
            use_temporal_bias=cfg['use_temporal_bias'],
            use_spatial_bias=cfg['use_spatial_bias'],
            use_history_self_attn=cfg['use_history_self_attn'],
            lat_mean=lat_mean, lon_mean=lon_mean,
            lat_std=lat_std, lon_std=lon_std
        )

        ablation_model, _, best_hr10 = train_model(
            ablation_model, train_loader, test_loader, ablation_config, device, verbose=False
        )

        final_metrics = evaluate(ablation_model, test_loader, device, config['eval_ks'])
        results.append({'name': cfg['name'], 'metrics': final_metrics})
        print(f"Results: {format_metrics(final_metrics)}")

        del ablation_model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return results

print("\n" + "#"*80)
print("ABLATION STUDY (Table 3)")
print("#"*80)

ablation_results = run_ablation_study(ABLATION_CONFIGS, data, CONFIG, device, num_epochs=15 if not QUICK_MODE else 5)


################################################################################
ABLATION STUDY (Table 3)
################################################################################

Ablation: Full Model


Training:   0%|          | 0/747 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

Training:   0%|          | 0/747 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

Training:   0%|          | 0/747 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/934 [00:00<?, ?it/s]

Results: HR@10: 63.81% | HR@20: 70.46% | HR@5: 54.01% | MRR: 34.97% | NDCG@10: 41.28% | NDCG@20: 42.98% | NDCG@5: 38.08%


In [18]:
# Section 16: Candidate Pool Size Experiment (Figure 3)

def run_candidate_pool_experiment(model, data, config, device, pool_sizes=[50, 100, 200, 500]):
    """Test performance with different candidate pool sizes."""
    results = {}

    for pool_size in pool_sizes:
        print(f"Testing candidate pool size: {pool_size}")
        test_loader_pool = get_dataloader(
            data, 'test', config['batch_size'], config['max_history_len'],
            num_negatives=pool_size - 1, candidate_pool_size=pool_size
        )
        metrics = evaluate(model, test_loader_pool, device, config['eval_ks'])
        results[pool_size] = metrics
        print(f"  HR@10: {metrics['HR@10']*100:.2f}%")

    return results

print("\n" + "#"*80)
print("CANDIDATE POOL SIZE EXPERIMENT (Figure 3)")
print("#"*80)

pool_results = run_candidate_pool_experiment(model, data, CONFIG, device, CONFIG['candidate_pool_sizes'])


################################################################################
CANDIDATE POOL SIZE EXPERIMENT (Figure 3)
################################################################################
Testing candidate pool size: 50
[test] 14940 samples, 2590 valid POI locations, eval: pool=50


Evaluating:   0%|          | 0/234 [00:00<?, ?it/s]

  HR@10: 92.48%
Testing candidate pool size: 100
[test] 14940 samples, 2590 valid POI locations, eval: pool=100


Evaluating:   0%|          | 0/234 [00:00<?, ?it/s]

  HR@10: 89.03%
Testing candidate pool size: 200
[test] 14940 samples, 2590 valid POI locations, eval: pool=200


Evaluating:   0%|          | 0/234 [00:00<?, ?it/s]

  HR@10: 85.15%
Testing candidate pool size: 500
[test] 14940 samples, 2590 valid POI locations, eval: pool=500


Evaluating:   0%|          | 0/234 [00:00<?, ?it/s]

  HR@10: 80.25%


In [ ]:
# Section 17: Robustness Analysis (Figure 4)

def count_poi_visits(train_data):
    poi_counts = defaultdict(int)
    for traj in train_data.values():
        for visit in traj:
            poi_counts[visit['poi_idx']] += 1
    return poi_counts

def analyze_robustness(model, data, config, device):
    """Analyze model robustness on sparse users and cold POIs (full POI evaluation)."""
    results = {}

    # User activity analysis
    train_data = data['train_data']
    user_lengths = {u: len(traj) for u, traj in train_data.items()}

    sparse_users = [u for u, l in user_lengths.items() if l < 50]
    active_users = [u for u, l in user_lengths.items() if l >= 50]

    print(f"Sparse users (<50 check-ins): {len(sparse_users)}")
    print(f"Active users (>=50 check-ins): {len(active_users)}")

    # POI popularity analysis
    poi_counts = count_poi_visits(train_data)
    cold_pois = set(p for p, c in poi_counts.items() if c < 20)
    hot_pois = set(p for p, c in poi_counts.items() if c >= 20)

    print(f"Cold POIs (<20 visits): {len(cold_pois)}")
    print(f"Hot POIs (>=20 visits): {len(hot_pois)}")

    # Evaluate on subsets (full POI evaluation)
    test_data = data['test_data']

    def create_subset_data(user_subset):
        subset = data.copy()
        subset['test_data'] = {u: traj for u, traj in test_data.items() if u in user_subset}
        return subset

    eval_bs = config.get('eval_batch_size', 16)

    if sparse_users:
        sparse_data = create_subset_data(set(sparse_users))
        sparse_loader = get_dataloader(sparse_data, 'test', eval_bs, config['max_history_len'],
                                        config['num_negatives'], candidate_pool_size=None)
        if len(sparse_loader) > 0:
            results['sparse_users'] = evaluate(model, sparse_loader, device, config['eval_ks'])
            print(f"Sparse users HR@10: {results['sparse_users']['HR@10']*100:.2f}%")

    if active_users:
        active_data = create_subset_data(set(active_users))
        active_loader = get_dataloader(active_data, 'test', eval_bs, config['max_history_len'],
                                        config['num_negatives'], candidate_pool_size=None)
        if len(active_loader) > 0:
            results['active_users'] = evaluate(model, active_loader, device, config['eval_ks'])
            print(f"Active users HR@10: {results['active_users']['HR@10']*100:.2f}%")

    return results

print("\n" + "#"*80)
print("ROBUSTNESS ANALYSIS (Figure 4)")
print("#"*80)

robustness_results = analyze_robustness(model, data, CONFIG, device)

In [20]:
# Section 18: Efficiency Evaluation (Table 4)

def evaluate_efficiency(model, test_loader, device, num_runs=100):
    """Evaluate model efficiency: latency, throughput, memory."""
    model.eval()
    batch = next(iter(test_loader))
    batch = {k: v.to(device) for k, v in batch.items()}
    batch_size = batch['history_pois'].shape[0]

    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(
                history_pois=batch['history_pois'],
                history_timestamps=batch['history_timestamps'],
                history_locations=batch['history_locations'],
                history_len=batch['history_len'],
                candidates=batch['candidates'],
                candidate_locations=batch['candidate_locations']
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # Measure latency
    latencies = []
    for _ in range(num_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            _ = model(
                history_pois=batch['history_pois'],
                history_timestamps=batch['history_timestamps'],
                history_locations=batch['history_locations'],
                history_len=batch['history_len'],
                candidates=batch['candidates'],
                candidate_locations=batch['candidate_locations']
            )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        latencies.append((time.time() - start) * 1000)

    # Memory
    if torch.cuda.is_available():
        memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
    else:
        memory_mb = 0

    avg_latency = np.mean(latencies)
    throughput = batch_size / (avg_latency / 1000)

    return {
        'latency_ms': avg_latency,
        'throughput_samples_per_sec': throughput,
        'memory_mb': memory_mb,
        'batch_size': batch_size
    }

print("\n" + "#"*80)
print("EFFICIENCY EVALUATION (Table 4)")
print("#"*80)

efficiency_results = evaluate_efficiency(model, test_loader, device)
print(f"Latency: {efficiency_results['latency_ms']:.2f} ms")
print(f"Throughput: {efficiency_results['throughput_samples_per_sec']:.1f} samples/sec")
print(f"Memory: {efficiency_results['memory_mb']:.1f} MB")


################################################################################
EFFICIENCY EVALUATION (Table 4)
################################################################################
Latency: 29.14 ms
Throughput: 549.0 samples/sec
Memory: 569.8 MB


---
## Part 8: Visualization & Summary

In [ ]:
# Section 19: Visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training curves
ax = axes[0, 0]
ax.plot(history['train_loss'], 'b-', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Curve')
ax.grid(True, alpha=0.3)

# 2. Ablation Study
ax = axes[0, 1]
names = [r['name'] for r in ablation_results]
hr10_values = [r['metrics']['HR@10'] * 100 for r in ablation_results]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
bars = ax.bar(range(len(names)), hr10_values, color=colors[:len(names)])
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('HR@10 (%)')
ax.set_title('Ablation Study (Table 3)')
for i, v in enumerate(hr10_values):
    ax.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontsize=8)

# 3. Candidate Pool Size
ax = axes[1, 0]
pool_sizes = list(pool_results.keys())
pool_hr10 = [pool_results[p]['HR@10'] * 100 for p in pool_sizes]
ax.plot(pool_sizes, pool_hr10, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Candidate Pool Size')
ax.set_ylabel('HR@10 (%)')
ax.set_title('Performance vs Pool Size (Figure 3)')
ax.grid(True, alpha=0.3)

# 4. Metrics over training
ax = axes[1, 1]
epochs = range(1, len(history['metrics']) + 1)
hr5 = [m['HR@5'] * 100 for m in history['metrics']]
hr10 = [m['HR@10'] * 100 for m in history['metrics']]
ax.plot(epochs, hr5, 'g-', linewidth=2, label='HR@5')
ax.plot(epochs, hr10, 'b-', linewidth=2, label='HR@10')
ax.set_xlabel('Epoch')
ax.set_ylabel('Hit Rate (%)')
ax.set_title('Evaluation Metrics Over Training')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved to 'experiment_results.png'")

In [22]:
# Section 20: Summary & Results

# Collect all results
all_experiment_results = {
    'dataset': DATASET,
    'timestamp': datetime.now().isoformat(),
    'main_results': final_metrics,
    'ablation_results': [{'name': r['name'], 'metrics': r['metrics']} for r in ablation_results],
    'pool_size_results': {str(k): v for k, v in pool_results.items()},
    'robustness_results': robustness_results,
    'efficiency_results': efficiency_results,
    'config': CONFIG,
    'data_stats': data['stats']
}

# Save results to JSON file
result_file = save_experiment_results(DATASET, all_experiment_results)

# Print formatted table
print_results_table(final_metrics, ablation_results, DATASET)

print(f"\n{'='*80}")
print("CaST-POI EXPERIMENT SUMMARY")
print("="*80)

print(f"\n[Dataset: {data['stats']['dataset']}]")
print(f"  Users: {data['stats']['num_users']}")
print(f"  POIs: {data['stats']['num_pois']}")
print(f"  Check-ins: {data['stats']['num_checkins']}")

print(f"\n[Model Configuration]")
print(f"  Embedding dim: {CONFIG['embed_dim']}")
print(f"  Attention heads: {CONFIG['num_heads']}")
print(f"  Encoder layers: {CONFIG['num_layers']}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\n[Main Results]")
print(f"  {format_metrics(final_metrics)}")

print(f"\n[Ablation Study (Table 3)]")
print(f"{'Configuration':<30} {'HR@10':>12} {'Change':>12}")
print("-"*55)
full_hr10 = ablation_results[0]['metrics']['HR@10']
for result in ablation_results:
    hr10 = result['metrics']['HR@10']
    change = '-' if result['name'] == 'Full Model' else f"{(hr10 - full_hr10) / full_hr10 * 100:+.2f}%"
    print(f"{result['name']:<30} {hr10*100:>11.2f}% {change:>12}")

print(f"\n[Candidate Pool Size (Figure 3)]")
for pool_size, metrics in pool_results.items():
    print(f"  Pool size {pool_size}: HR@10 = {metrics['HR@10']*100:.2f}%")

print(f"\n[Efficiency (Table 4)]")
print(f"  Latency: {efficiency_results['latency_ms']:.2f} ms")
print(f"  Throughput: {efficiency_results['throughput_samples_per_sec']:.1f} samples/sec")
print(f"  Memory: {efficiency_results['memory_mb']:.1f} MB")

print(f"\n{'='*80}")
print(f"Results saved to: {result_file}")
print(f"Experiment completed successfully for {DATASET.upper()}!")
print("="*80)


CaST-POI EXPERIMENT SUMMARY

[Dataset: NYC]
  Users: 770
  POIs: 2591
  Check-ins: 68305

[Model Configuration]
  Embedding dim: 64
  Attention heads: 8
  Encoder layers: 2
  Parameters: 501,315

[Main Results]
  HR@10: 67.09% | HR@20: 73.24% | HR@5: 56.09% | MRR: 35.98% | NDCG@10: 42.89% | NDCG@20: 44.46% | NDCG@5: 39.29%

[Ablation Study (Table 3)]
Configuration                         HR@10       Change
-------------------------------------------------------
Full Model                           62.12%            -
w/o Temporal Bias                    63.09%       +1.56%
w/o Spatial Bias                     59.00%       -5.02%
w/o Both Biases                      58.45%       -5.92%
w/o History Self-Attn                63.81%       +2.72%

[Candidate Pool Size (Figure 3)]
  Pool size 50: HR@10 = 92.48%
  Pool size 100: HR@10 = 89.03%
  Pool size 200: HR@10 = 85.15%
  Pool size 500: HR@10 = 80.25%

[Efficiency (Table 4)]
  Latency: 29.14 ms
  Throughput: 549.0 samples/sec
  Memory: 5